In [7]:
import pandas as pd
import os
import glob

In [8]:
# --- USER CONFIGURATION ---
INPUT_FILE_PATH = r'D:\ev_data\outputs\ev_ouput_data\us_ev.csv'  # Replace with your actual csv file path
OUTPUT_FOLDER = 'D:\ev_data\outputs\ev_ouput_data\ev-main'       # Folder to store the chunks
CHUNK_SIZE = 1000                        # Number of rows per small csv

In [9]:
import math

# 1. Create output directory if it doesn't exist
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created folder: {OUTPUT_FOLDER}")

# 2. Read the dataframe
print(f"Reading file: {INPUT_FILE_PATH}")
df = pd.read_csv(INPUT_FILE_PATH)

# 3. Add an index column to track original order
# This ensures we can put the rows back in the exact same order later
df['restore_index'] = range(len(df))

# 4. Calculate total chunks needed
total_rows = len(df)
num_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"Splitting {total_rows} rows into {num_chunks} files...")

# 5. Loop and Write
for i in range(num_chunks):
    start_row = i * CHUNK_SIZE
    end_row = start_row + CHUNK_SIZE
    
    # Slice the dataframe
    chunk = df.iloc[start_row:end_row]
    
    # Generate filename (e.g., chunk_0.csv, chunk_1.csv)
    # We use zfill(5) to make filenames like chunk_00001.csv for better sorting
    filename = f"chunk_{str(i).zfill(5)}.csv"
    file_path = os.path.join(OUTPUT_FOLDER, filename)
    
    # Write to disk
    chunk.to_csv(file_path, index=False)
    
    # Optional: Print progress every 10 chunks to avoid cluttering screen
    if i % 10 == 0:
        print(f"Written {filename}...")

print("✅ All small CSVs have been written to the output folder.")

Reading file: D:\ev_data\outputs\ev_ouput_data\us_ev.csv
Splitting 71460 rows into 72 files...
Written chunk_00000.csv...
Written chunk_00010.csv...
Written chunk_00020.csv...
Written chunk_00030.csv...
Written chunk_00040.csv...
Written chunk_00050.csv...
Written chunk_00060.csv...
Written chunk_00070.csv...
✅ All small CSVs have been written to the output folder.


In [10]:
print("Looking for files to merge...")

# 1. Get list of all CSV files in the output folder
# We sort them to try and load them in rough order, though 'restore_index' is the ultimate safety
all_files = sorted(glob.glob(os.path.join(OUTPUT_FOLDER, "chunk_*.csv")))

print(f"Found {len(all_files)} files. Merging now...")

# 2. Read each file and add to a list
df_list = []
for filename in all_files:
    # Read the individual small csv
    df_temp = pd.read_csv(filename)
    df_list.append(df_temp)

# 3. Combine into one big dataframe
merged_df = pd.concat(df_list, ignore_index=True)

# 4. CRITICAL STEP: Sort by the 'restore_index' we added earlier
# This fixes any rows that might have been loaded out of order
merged_df = merged_df.sort_values('restore_index')

# 5. Remove the helper column so the file looks exactly like the original
merged_df = merged_df.drop(columns=['restore_index'])

# 6. Save the merged file (Optional)
merged_output_path = os.path.join(OUTPUT_FOLDER, 'fully_merged_file.csv')
merged_df.to_csv(merged_output_path, index=False)

print(f"✅ Merge complete!")
print(f"Total rows in merged file: {len(merged_df)}")
print(f"Merged file saved to: {merged_output_path}")

Looking for files to merge...
Found 72 files. Merging now...
✅ Merge complete!
Total rows in merged file: 71460
Merged file saved to: D:\ev_data\outputs\ev_ouput_data\ev-main\fully_merged_file.csv
